# Algorytm RSA

Obecnie najpopularniejszy algorytm kryptografi asymetrycznej. Opublikowany w 1977 roku w *Scientific American's Mathematical Games* przez Rona Rivesta, Adiego Shamira i Leonarda Adlemana. Opiera się na problemie faktoryzacji liczb.

W uproszczonej wersji algorytm RSA wygląda następująco:

### Generowanie kluczy

- Wybieramy dwie losowe liczby pierwsze $p$ i $q$
- Obliczamy $n=pq$
- Obliczamy funkcję Eulera $\phi(n)=(p-1)(q-1)$
- Wybieramy $e\in\{2,...,\phi(n)-1\}$ względnie pierwszą z $\phi(n)$
- Obliczamy $d=e^{-1}\mod \phi(n)$
- Klucz publiczny to para $(n,e)$ a klucz prywatny to para $(n,d)$.


### Szyfrowanie
Mamy dany klucz publiczny $(n,e)$. Wiadomość szyfrowana jest liczbą $m\in [0,n)$.
- Obliczamy szyfrogram $$c=m^e\mod n.$$

Teoretycznie może się zdarzyć, że szyfrogram $c$ jest taki sam, jak szyfrowana wiadomość $m$.

### Deszyfrowanie
- Obliczamy odszyfrowaną wiadomość $$m=c^d=m^{ed}\mod n$$

### Ćwiczenie 1.

Zaimplementuj uproszczony algorytm RSA.

In [1]:
# TYPE YOUR CODE BELOW
import math
import random


def generate_keypair(p, q, e=None):
    """
    Generates an RSA public and private key pair.
    Assumes p and q are prime numbers.
    """
    # Calculate n
    n = p * q

    # Calculate Euler's totient function phi(n)
    phi = (p - 1) * (q - 1)

    # Choose e such that 1 < e < phi and e is coprime to phi
    if e is None:
        e = random.randrange(2, phi)
        while math.gcd(e, phi) != 1:
            e = random.randrange(2, phi)

    assert 1 < e < phi, "Choose e such that 1 < e < phi"
    assert math.gcd(e, phi) == 1, "Choose e such that e is coprime to phi"

    # Calculate d, the modular inverse of e modulo phi
    d = pow(e, -1, phi)

    public_key = (n, e)
    private_key = (n, d)

    return public_key, private_key


def encrypt(public_key, m):
    """
    Encrypts an integer message m using the public key.
    The message m must be in the range [0, n-1].
    """
    n, e = public_key
    # c = m^e mod n
    return pow(m, e, n)


def decrypt(private_key, c):
    """
    Decrypts a ciphertext c using the private key.
    """
    n, d = private_key
    # m = c^d mod n
    return pow(c, d, n)

In [2]:
p = 61
q = 53

# Generate keys
pub_key, priv_key = generate_keypair(p, q)
print(f"Public Key (n, e): {pub_key}")
print(f"Private Key (n, d): {priv_key}")

original_message = 123
print(f"\nOriginal Message: {original_message}")

# Encrypt
encrypted_msg = encrypt(pub_key, original_message)
print(f"Encrypted Message: {encrypted_msg}")

# Decrypt
decrypted_msg = decrypt(priv_key, encrypted_msg)
print(f"Decrypted Message: {decrypted_msg}")

Public Key (n, e): (3233, 7)
Private Key (n, d): (3233, 1783)

Original Message: 123
Encrypted Message: 2868
Decrypted Message: 123


# Proste ataki na RSA

## Faktoryzacja Fermata

Faktoryzacja Fermata liczby nieparzystej $n$ opiera się na znalezieniu pary liczb $a,b$ takich, że $n=a^2-b^2$. Wtedy od razu otrzymujemy faktoryzację $$n=(a+b)(a-b).$$

Dla dowolnej złożonej liczby nieparzystej (tzn. liczby nieparzystej nie będącej liczbą pierwszą) znajdziemy takie liczby $a,b$. W szczególności, w przypadku $n=pq$ (jak w RSA) mamy $$a=\frac{p+q}{2},\qquad b=\frac{p-q}{2}.$$

#### Krok 0.
Sprawdzamy, czy $\sqrt{n}$ jest liczbą naturalną. Jeżeli tak - znaleźliśmy faktoryzację i koniec algorytmu.
#### Krok 1.
Definiujemy zmienne pomocnicze
$$a = \left\lceil\sqrt{n}\right\rceil,\qquad
b^2 = a^2 - n.$$
#### Krok 2.
Jeżeli $\sqrt{b^2}$ jest liczbą całkowitą, to kończymy algorytm i zwracamy $a$ oraz $b=\sqrt{b^2}$. Jeżeli nie, to aktualizujemy zmienne:
$$a=a + 1,\qquad b^2=a^2 - n.$$


### Ćwiczenie 2.
Zaimplementuj atak z użyciem faktoryzacji Fermata i przetestuj go na swojej implementacji RSA. Jeżeli atak zakończył się sukcesem - popraw implementację kryptosystemu. Jakie założenia powinny spełniać parametry kryptosystemu, aby atak się nie powiódł?

In [3]:
def fermat_factorization(n, max_iters=float("inf")):
    """
    Performs Fermat's factorization on an odd composite number n.
    Returns a tuple of prime factors (p, q).
    """
    # Step 0: Check if sqrt(n) is an integer
    a = math.isqrt(n)
    if a * a == n:
        return (a, a)

    # Step 1: Initialize variables
    a += 1
    b_sq = a * a - n

    # Step 2: Loop until b_sq is a perfect square
    while max_iters := max_iters - 1:
        b = math.isqrt(b_sq)
        if b * b == b_sq:
            # Perfect square found, return factors (a-b) and (a+b)
            return (a - b, a + b)

        # Update variables if not a perfect square
        a += 1
        b_sq = a * a - n

In [4]:
import time

ns_weak = [
    2101644002566781,
    3593875791313441,
    800090608581732401,
    22601441855002489679,
]

for n_weak in ns_weak:
    print(f"Target modulus (n): {n_weak}")

    # Perform the attack
    start_time = time.perf_counter()
    p_found, q_found = fermat_factorization(n_weak)
    elapsed = time.perf_counter() - start_time
    print(f"Found factors in {elapsed*1000:.2f} ms: p = {p_found}, q = {q_found}\n")

Target modulus (n): 2101644002566781
Found factors in 1160.91 ms: p = 21341587, q = 98476463

Target modulus (n): 3593875791313441
Found factors in 443.37 ms: p = 39269227, q = 91518883

Target modulus (n): 800090608581732401
Found factors in 0.30 ms: p = 891957881, q = 897004921

Target modulus (n): 22601441855002489679
Found factors in 1.04 ms: p = 4744066541, q = 4764149419



In [5]:
def fermat_attack(pub_key, encrypted_msg, max_iters):
    n, e = pub_key

    result = fermat_factorization(n, max_iters)
    
    if result is None:
        print("Attack failed")
        return
    
    # If we successfully found p and q, we can decrypt the message
    p, q = result
    
    # 1. Calculate Euler's totient function phi(n)
    phi = (p - 1) * (q - 1)
    
    # 2. Calculate the private exponent d
    d = pow(e, -1, phi)
    
    # 3. Decrypt the message using d
    decrypted_msg = pow(encrypted_msg, d, n)
    
    return decrypted_msg

In [6]:
fermat_attack(pub_key, encrypted_msg, 1_000_000)

123

# Ataki nie wykorzystujące faktoryzacji

## Atak naiwny

Jeżeli wykładnik $e$ jest niewielki oraz dla wiadomości $m$ wartość $m^e<n$, to $c=m^e\mod n=m^e$. Możemy zatem odzyskać $m$ obliczając $m=\sqrt[e]{c}$.

### Ćwiczenie 3.
Sprawdź, czy Twoja implementacja RSA jest podatna na powyższy atak wykonując co najmniej 1000 prób. Jeżeli którakolwiek zakończyła się powodzeniem - popraw implementację. Jakie założenia powinny spełniać parametry kryptosystemu, żeby atak nie miał szans powodzenia?

In [7]:
# TYPE YOUR CODE BELOW

def naive_attack(pub_key, encrypted_msg, max_iters):
    """
    Attempts to recover the message by calculating the e-th root of the ciphertext.
    Works only for textbook RSA when m^e < n.
    """
    n, e = pub_key
    
    # Binary search for the exact integer e-th root
    low, high = 0, encrypted_msg
    
    while low <= high and max_iters > 0:
        max_iters -= 1
        mid = (low + high) // 2
        val = pow(mid, e)
        
        if val == encrypted_msg:
            return mid  # Exact root found
        elif val < encrypted_msg:
            low = mid + 1
        else:
            high = mid - 1
            
    return None

In [8]:
# Setup vulnerable parameters (small public exponent e=3)
p_test, q_test = 7907, 7919 
pub_key_test, priv_key_test = generate_keypair(p_test, q_test, e=3)
n_test, e_test = pub_key_test

success_count = 0
attempts = 1000

for _ in range(attempts):
    # Pick a small message m such that m^e < n
    # For n = ~62.6 million and e = 3, max_m is around 390
    m_original = random.randint(2, 390)
    c_intercepted = encrypt(pub_key_test, m_original)
    
    # We pass a high max_iters to ensure binary search can finish
    recovered_m = naive_attack(pub_key_test, c_intercepted, max_iters=10000)
    
    if recovered_m == m_original:
        success_count += 1
        
print(f"Naive attack successes: {success_count} out of {attempts}")

Naive attack successes: 1000 out of 1000


## Atak Wienera

Atak Wienera odtwarza klucz prywatny $d$ z klucza publicznego $(n,e)$ bez konieczności faktoryzacji $n$.

#### Krok 1.
Przeksztacamy ułamek $\frac{e}{n}$ na ułamek łańcuchowy postaci $${\displaystyle x=a_{0}+{\cfrac {1}{a_{1}+{\cfrac {1}{a_{2}+{\cfrac {1}{a_{3}+{\cfrac {1}{a_{4}+\ddots \,\frac{1}{a_k}}}}}}}}}}$$o standardowej reprezentacji $[a_0;a_1,a_2,...,a_{k−2},a_{k−1},a_k]$.

#### Krok 2.
Dla każdego reduktu ułamka łańcuchowego, tzn. ułamka łańcuchowego postaci $[a_0]$, $[a_0;a_1]$, $[a_0;a_1,a_2]$,...,$[a_0;a_1,a_2,...,a_{k−2},a_{k−1},a_k]$ sprawdzamy, czy jest on rozwinięciem ułamka $\frac{s}{d}$ w ułamek łańcuchowy dla pewnej stałej $s$:
- niech $l$ oznacza licznik a $m$ - mianownik reduktu
- sprawdź, czy mianownik $m$ jest nieparzysty. Jeżeli nie - przejdź do sprawdzania kolejnego reduktu.
- sprawdź, czy $em=1\mod l$. Jeżeli nie - przejdź do sprawdzania kolejnego reduktu.
- zdefiniuj wielomian pomocniczy $$Q(x)=x^2-(n-\frac{em-1}{l}+1)x+n.$$Jeżeli pierwiastki tego wielomianu są liczbami całkowitymi, to aktualny mianownik $m$ reduktu jest szukanym kluczem prywatnym $d$.


Atak Wienera niekoniecznie musi zakończyć się sukcesem - jeżeli sprawdzimy wszystkie redukty i żaden nie jest rozwinięciem $\frac{s}{d}$, to dany zestaw parametrów RSA jest odporny na ten atak.

### Ćwiczenie 4.

Sprawdź, czy Twoja implementacja RSA jest podatna na atak Wienera wykonując co najmniej 100 prób. Jeżeli którakolwiek zakończyła się powodzeniem - popraw implementację. Jakie założenia powinny spełniać parametry kryptosystemu, żeby atak nie miał szans powodzenia?

In [9]:
def frac_to_cont(p, q):
    """
    funkcja przeliczająca ułamek zwykły p/q na postać łańcuchową a=[a0;a1,...,ak]
    zwraca listę a
    """
    a = []
    while q:
        a.append(p // q)
        p, q = q, p % q
    return a


def cont_to_frac(a):
    """
    funkcja przeliczająca ułamek łańcuchowy a=[a0;a1,...,ak] na ułamek zwykły p/q
    zwraca tuplę (p,q)
    """
    m = 1
    l = a[-1]
    for i in range(2, len(a) + 1):
        lo = l
        l = m
        m = lo
        l = a[-i] * m + l
    return (l, m)

In [10]:
# TYPE YOUR CODE BELOW

def wiener_attack(pub_key, encrypted_msg, max_iters=1000):
    """
    Attempts Wiener's attack to recover the private key d and decrypt the message.
    Returns the decrypted message if successful, None otherwise.
    """
    n, e = pub_key
    
    # Krok 1: Convert e/n to a continued fraction
    a = frac_to_cont(e, n)
    
    # Krok 2: Iterate through the convergents (redukty)
    # Check up to max_iters or the length of the continued fraction
    for k in range(1, min(len(a) + 1, max_iters + 1)):
        # Calculate numerator (l) and denominator (m) of the k-th convergent
        l, m = cont_to_frac(a[:k])
        
        # Rule 1: Denominator m must be odd (since it represents d)
        if m % 2 == 0:
            continue
            
        # Rule 2: e*m must be congruent to 1 modulo l
        if l == 0 or (e * m - 1) % l != 0:
            continue
            
        # Calculate the potential Euler's totient function phi(n)
        phi_candidate = (e * m - 1) // l
        
        # Rule 3: Define polynomial Q(x) = x^2 - Bx + n
        # where B = n - phi_candidate + 1
        B = n - phi_candidate + 1
        
        # Calculate discriminant (Delta = B^2 - 4n)
        delta = B * B - 4 * n
        
        if delta >= 0:
            sq_delta = math.isqrt(delta)
            # Check if delta is a perfect square
            if sq_delta * sq_delta == delta:
                # Check if roots (B +/- sq_delta) / 2 are integers
                # They are integers if the numerator is even
                if (B + sq_delta) % 2 == 0:
                    # Success! The denominator m is our private key d.
                    d = m
                    
                    # Decrypt the message using the recovered private key
                    return pow(encrypted_msg, d, n)
                    
    # Attack failed to find a valid d
    return None

In [11]:
def generate_vulnerable_rsa(p, q):
    """
    Generates an RSA keypair explicitly vulnerable to Wiener's attack.
    This requires choosing a very small private exponent d.
    """
    n = p * q
    phi = (p - 1) * (q - 1)
    
    # Wiener's attack upper bound for d
    limit = int((n ** 0.25) / 3)
    
    # Find a valid small d
    d = 3
    while d < limit:
        if math.gcd(d, phi) == 1:
            break
        d += 2
        
    # e is the modular inverse of d modulo phi
    e = pow(d, -1, phi)
    
    return (n, e), (n, d)

# Run the 100 attempts test
p_test = 10007
q_test = 10009
pub_key_vuln, priv_key_vuln = generate_vulnerable_rsa(p_test, q_test)

success_count = 0
attempts = 100

for _ in range(attempts):
    m_original = random.randint(2, pub_key_vuln[0] - 1)
    c_intercepted = encrypt(pub_key_vuln, m_original)
    
    recovered_m = wiener_attack(pub_key_vuln, c_intercepted)
    
    if recovered_m == m_original:
        success_count += 1
        
print(f"Wiener attack successes: {success_count} out of {attempts}")

Wiener attack successes: 100 out of 100
